# Setup

In [ ]:
import mlx.core as mx
from mlx_lm import load, stream_generate, generate, batch_generate
from mlx_lm.sample_utils import make_sampler
from mlx_lm.models.cache import make_prompt_cache
from mlx.utils import tree_flatten
from mlx_lm.models.cache import KVCache

from tqdm import tqdm
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
# KVCache.step = 32            # default 256


In [ ]:
print(f"{(mx.get_active_memory()) / 1e6:.1f} MB")

# Prefill + decode

## Prefill

In [ ]:
model, tok = load("mlx-community/Qwen3-4B-4bit")

In [ ]:
print(f"{(mx.get_active_memory()) / 1e6:.1f} MB")

In [ ]:

PROMPT = "ciao io sono Joe! tu come ti chiami?"

msgs = [{"role": "user", "content": PROMPT}]
prompt = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              enable_thinking=False,tokenize = True, return_tensors="mlx")


In [ ]:
print(prompt)

In [ ]:
prompt.shape

In [ ]:
before = mx.get_active_memory()
print(f"{(before) / 1e6:.1f} MB")

In [ ]:
cache = make_prompt_cache(model)
model(prompt, cache=cache)
mx.eval([c.state for c in cache])

In [ ]:
cache[3].keys.shape

In [ ]:
b, h, t, d = cache[0].keys.shape
print(f"layer con cache: {len(cache)}")
print(f"numero di richieste:    {b}")
print(f"kv heads: {h}")
print(f"token:    {t}  (usati {cache[0].offset}, il resto è blocco vuoto)")
print(f"head dim: {d}")

In [ ]:
mb = sum(a.nbytes for _, a in tree_flatten([c.state for c in cache])) / 1e6
print(f"{mb:.1f} MB")

In [ ]:
after = mx.get_active_memory()
print(f"{(after) / 1e6:.1f} MB")

In [ ]:
print(f"{(after - before) / 1e6:.1f} MB")

## Decode e ttft

In [ ]:
import time

PROMPT = "casa " * 2000 + "\nquante volte ho scritto casa?"
msgs = [{"role": "user", "content": PROMPT}]
prompt = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                 enable_thinking=False, return_tensors="mlx")
print(prompt.shape)

In [ ]:
cache = make_prompt_cache(model)
model(prompt[:, :-1], cache=cache)
mx.eval([c.state for c in cache])

In [ ]:
t0 = time.perf_counter()
generate(model, tok, prompt[0], max_tokens=1)
print(f"TTFT senza cache: {time.perf_counter() - t0:.2f} s")

In [ ]:
t0 = time.perf_counter()
generate(model, tok, prompt[0, -1:], max_tokens=1, prompt_cache=cache)
print(f"TTFT con cache: {time.perf_counter() - t0:.2f} s")

## Decode senza cache

In [ ]:
import time
import mlx.core as mx
import matplotlib.pyplot as plt
from mlx_lm.models.cache import make_prompt_cache

PROMPT = "Raccontami in dettaglio come funziona un motore a scoppio."
N = 100

msgs = [{"role": "user", "content": PROMPT}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False)

model(mx.array([ids[:4]])).item  # riscaldamento: il primo giro compila i kernel
mx.eval(model(mx.array([ids[:4]])))

In [ ]:
def senza_cache(ids, n):
    seq, out, tempi = list(ids), [], []
    for _ in range(n):
        t0 = time.perf_counter()
        logits = model(mx.array([seq]))              # ogni giro: tutta la frase da capo
        nxt = mx.argmax(logits[0, -1]).item()
        tempi.append(time.perf_counter() - t0)
        out.append(nxt)
        seq.append(nxt)
    return out, tempi


def con_cache(ids, n):
    cache = make_prompt_cache(model)
    x = mx.array([ids])                              # primo giro: tutto il prompt (prefill)
    out, tempi = [], []
    for _ in range(n):
        t0 = time.perf_counter()
        logits = model(x, cache=cache)
        nxt = mx.argmax(logits[0, -1]).item()
        tempi.append(time.perf_counter() - t0)
        out.append(nxt)
        x = mx.array([[nxt]])                        # dal secondo giro: solo il token nuovo
    return out, tempi

In [ ]:
out_s, t_s = senza_cache(ids, N)
out_c, t_c = con_cache(ids, N)

In [ ]:
print(f"senza cache: {sum(t_s):.1f} s  |  con cache: {sum(t_c):.1f} s")

plt.figure(figsize=(8, 4))
plt.plot([t * 1000 for t in t_s], label="senza cache", color="#E15759", lw=2)
plt.plot([t * 1000 for t in t_c], label="con cache", color="#59A14F", lw=2)
plt.xlabel("token generati")
plt.ylabel("ms per token")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Decode streaming

In [ ]:
PROMPT = "Scrivi una funzione Python che legge un CSV, raggruppa per una colonna e calcola media e deviazione standard di un'altra. Commenta ogni passaggio."
msgs = [{"role": "user", "content": PROMPT}]
prompt = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              enable_thinking=False,tokenize = True)
len(prompt)

In [ ]:
test = stream_generate(model, tok, prompt, max_tokens=300, sampler=make_sampler(temp=0.0))

In [ ]:
answer = ""

In [ ]:
next_token = next(test)
# print(next_token.text, next_token.generation_tokens)
answer+= next_token.text
print(answer)
print(next_token.generation_tokens)

In [ ]:
max_tokens = 10_000
answer = ""
for r in stream_generate(model, tok, prompt, max_tokens=max_tokens,
                         sampler=make_sampler(temp=0.0), prefill_step_size=256):
    answer += r.text
    clear_output(wait=True)
    print(answer)
    print(f"\n──────── tokens left: {max_tokens - r.generation_tokens}" f" | {r.generation_tps:.1f} tok/s" f" | picco {r.peak_memory:.2f} GB ──────────")

# Cache visualization

In [ ]:
max_tokens = 1000
base = mx.get_active_memory()
remaining, tps, kv_mb = [], [], []

for r in stream_generate(model, tok, prompt, max_tokens=max_tokens,
                         sampler=make_sampler(temp=0.0)):
    remaining.append(max_tokens - r.generation_tokens)
    tps.append(r.generation_tps)
    kv_mb.append((mx.get_active_memory() - base) / 1e6)

    if r.generation_tokens % 100:
        continue

    clear_output(wait=True)
    fig, ax = plt.subplots(3, 1, figsize=(7, 6), sharex=True)

    series = [
        (remaining,  "tokens left", "#4E79A7", f"{remaining[-1]}"),
        (tps[5:],  "tok/s",         "#59A14F", f"{tps[-1]:.1f}"),
        (kv_mb,       "KV cache MB",   "#E15759", f"{kv_mb[-1]:.0f}"),
    ]

    for a, (values, label, color, text) in zip(ax, series):
        a.plot(values, color=color, lw=2)
        a.set_ylabel(label)
        a.text(0.99, 0.92, text, transform=a.transAxes,
               ha="right", va="top", fontsize=14, fontweight="bold",
               color="#333333")
        a.grid(alpha=0.25)
        a.spines[["top", "right"]].set_visible(False)

    ax[2].set_xlabel("generated tokens")
    plt.tight_layout()
    plt.show()

# Parallel prompts

In [ ]:
questions = ["Spiega cos'è un CSV.", "Scrivi una funzione che somma una lista.",
           "Cos'è la deviazione standard?", "Differenza tra media e mediana?"]
prompts = [tok.apply_chat_template([{"role": "user", "content": d}],
                                   add_generation_prompt=True, enable_thinking=False)
           for d in questions]



In [ ]:
t0 = time.perf_counter()
for p in prompts:
    generate(model, tok, p, max_tokens=200, sampler=make_sampler(temp=0.0))
print(f"sequential: {time.perf_counter() - t0:.1f} s")

t0 = time.perf_counter()
r = batch_generate(model, tok, prompts, max_tokens=200)
print(f"batched: {time.perf_counter() - t0:.1f} s")